In [ ]:
## An Introduction to Qiskit and Aer

# Objectives
'''
  - Understand how quantum circuits map to linear algebra
  - Build and simulate circuits in Qiskit
  - Compare statevector vs shot-based simulation
  - Model noise using Aer
  - Understand scaling and hardware constraints
'''

In [ ]:
# Install Qiskit and Necessary imports

!pip install qiskit qiskit-aer --quiet
!pip install pylatexenc --quiet


from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Single Qubit Superposition Circuit

qc = QuantumCircuit(1) # Default state is ket_0
qc.h(0)
#qc.h(2)
qc.measure_all()
qc.draw("mpl")

In [ ]:
# Run Shot-based simulations

backend = AerSimulator()
qc_t = transpile(qc, backend)

result = backend.run(qc_t, shots=10000).result()
counts = result.get_counts()

print("Measurement Counts:", counts)

In [ ]:
# Exact Statevector Simulation

qc_sv = QuantumCircuit(1)
qc_sv.h(0)

state = Statevector.from_instruction(qc_sv)
print(state)
  # No sampling noise
  # Exact amplitudes

In [ ]:
# Entanglement - Creating a Bell State

qc_bell = QuantumCircuit(2)
qc_bell.h(0)
qc_bell.cx(0,1)
qc_bell.draw("mpl")

In [ ]:
# The Bell Statevector

bell_state = Statevector.from_instruction(qc_bell)
print(bell_state)

In [ ]:
# Shot simulation of the Bell State

qc_bell.measure_all()
qc_bell_t = transpile(qc_bell, backend)

result = backend.run(qc_bell_t, shots=10000).result()
print(result.get_counts())

In [ ]:
# Scaling Problem: Memory Scaling Calculation

def memory_required(n):
    return (2**n) * 16 / (1024**3)

for n in [10, 20, 30, 50]:
    print(f"{n} qubits requires approx {memory_required(n):.2f} GB")

# The exponential scaling: 2^𝑛 × 16 bytes

In [ ]:
# Noise Modeling
# Depolarizing Noise

from qiskit_aer.noise import NoiseModel, depolarizing_error

noise_model = NoiseModel()
error = depolarizing_error(0.1, 1)

noise_model.add_all_qubit_quantum_error(error, ['h'])

qc_noise = QuantumCircuit(1)
qc_noise.h(0)
qc_noise.measure_all()

qc_noise_t = transpile(qc_noise, backend)

result_noise = backend.run(
    qc_noise_t,
    noise_model=noise_model,
    shots=1000
).result()

print("Noisy counts:", result_noise.get_counts())

# With this noise, density matrix changes to: ρ → (1−p)*ρ + p*I​/2

In [ ]:
# Transpilation and Optimization
# Optimization Levels

qc_complex = QuantumCircuit(2)
qc_complex.h(0)
qc_complex.cx(0,1)
qc_complex.cx(0,1)

qc_opt0 = transpile(qc_complex, backend, optimization_level=0)
qc_opt3 = transpile(qc_complex, backend, optimization_level=3)

print("Depth without optimization:", qc_opt0.depth())
print("Depth with optimization:", qc_opt3.depth())

In [ ]:
# Summary: What We Learned So far
'''
    - Circuits implement unitary evolution
    - Measurement implements Born rule sampling
    - Statevector simulation gives exact amplitudes
    - Shot simulation mimics real hardware
    - Noise reduces coherence
    - Classical simulation scales exponentially
'''